In [1]:
from pathlib import Path
import pandas as pd
from sliderule import sliderule, earthdata

SITES_PATH = Path("../../../scratch/climate_site_selection/regional_site_selection/candidates_d8bbbaa5ea8a7a28/selection_e5c0ae68a48ece02/n09/selected_sites.csv")

site = pd.read_csv(SITES_PATH).iloc[0]
west, south, east, north = map(
    float, [site.west, site.south, site.east, site.north]
)

polygon = [
    {"lon": west, "lat": south},
    {"lon": east, "lat": south},
    {"lon": east, "lat": north},
    {"lon": west, "lat": north},
    {"lon": west, "lat": south},
]

print(f"Site: {site.site_id}")
print(f"Bounds: {west}, {south}, {east}, {north}")

sliderule.init(verbose=False, rethrow=True)

resources = earthdata.cmr(
    short_name="ATL03",
    version="007",
    polygon=polygon,
    time_start="2020-01-01T00:00:00Z",
    time_end="2020-05-01T00:00:00Z",
)

print(f"Discovery succeeded: {len(resources)} resources")
print(*resources[:10], sep="\n")

Site: S01
Bounds: -126.6, 61.1, -118.6, 64.8


Warning, this environment is using an outdated client (v5.6.3). The code will run but some functionality supported by the server (v5.8.0) may not be available.


Discovery succeeded: 93 resources
ATL03_20200322223406_13320603_007_01.h5
ATL03_20200426092105_04710705_007_01.h5
ATL03_20200127131539_04860605_007_01.h5
ATL03_20200110141434_02270605_007_01.h5
ATL03_20200203132436_05930605_007_01.h5
ATL03_20200328104458_00290705_007_01.h5
ATL03_20200327220004_00210703_007_01.h5
ATL03_20200220122537_08520605_007_01.h5
ATL03_20200109015520_02040603_007_01.h5
ATL03_20200429202750_05240703_007_01.h5


In [6]:
from pathlib import Path
from copy import deepcopy
import sys
import yaml
import pandas as pd
from IPython.display import display
from sliderule import sliderule, earthdata

SCRIPT_DIR = Path(
    "/home/s1803229/Chapter_2/scripts/17-tl-processing-ATL08-remote-access"
)
CONFIG_PATH = Path("/home/s1803229/Chapter_2/config.yaml")
SITES_PATH = Path(
    "../../../scratch/climate_site_selection/"
    "regional_site_selection/candidates_d8bbbaa5ea8a7a28/"
    "selection_e5c0ae68a48ece02/n09/selected_sites.csv"
)
OUTPUT_DIR = Path("../../../scratch/s1803229/ATL08_parquet_test")

SITE_INDEX = 0
START = "2020-01-01T00:00:00Z"
END = "2020-08-01T00:00:00Z"
MAX_OVERPASSES = 30
LENGTH_M = 30
BOX_SIDE_KM = 1

# Check paths before starting remote requests.
for path in [SCRIPT_DIR / "remote_batch_core.py", CONFIG_PATH, SITES_PATH]:
    if not path.is_file():
        raise FileNotFoundError(path)

sys.path.insert(0, str(SCRIPT_DIR))
import remote_batch_core as remote

with CONFIG_PATH.open() as stream:
    cfg = yaml.safe_load(stream)

cfg["parallel_blocks"].update(
    rebinned=LENGTH_M,
    small_box_half_side=BOX_SIDE_KM / 2,
    DW=False,
    trim_atmospheric=True,
    sat_flag=True,
    msw_flag_threshold=float("inf"),
    layer_flag_threshold=float("inf"),
)

sites = remote.load_sites(SITES_PATH)
site = next(sites.iloc[[SITE_INDEX]].itertuples(index=False))

polygon = [
    {"lon": float(lon), "lat": float(lat)}
    for lon, lat in [
        (site.west, site.south),
        (site.east, site.south),
        (site.east, site.north),
        (site.west, site.north),
        (site.west, site.south),
    ]
]

sliderule.init(verbose=False, rethrow=True)

print(f"Discovering overpasses for {site.site_id}: {START} to {END}", flush=True)
resources = earthdata.cmr(
    short_name="ATL03",
    version="007",
    polygon=polygon,
    time_start=START,
    time_end=END,
)

# Keep the latest revision of each acquisition.
latest = {}
for resource in sorted(set(resources)):
    latest[resource.rsplit("_", 1)[0]] = resource

chosen = sorted(latest.values())[:MAX_OVERPASSES]
print(f"Found {len(latest)} acquisitions; testing {len(chosen)}.", flush=True)
if not chosen:
    raise RuntimeError("No acquisitions found in this date range.")

provider = remote.public_provider()
frames, diagnostics = [], []

for i, resource in enumerate(chosen, 1):
    print(f"\nOverpass {i}/{len(chosen)}: {resource}", flush=True)
    try:
        run = remote.process_resource(
            site, resource, deepcopy(cfg), provider, graph_detail=0
        )
        diagnostics.append(run.diagnostics.assign(resource=resource))
        display(run.diagnostics)
        print(f"Retained rows: {len(run.results):,}", flush=True)
        if not run.results.empty:
            frames.append(run.results)
    except Exception as exc:
        print(f"FAILED: {type(exc).__name__}: {exc}", flush=True)

diagnostics = (
    pd.concat(diagnostics, ignore_index=True)
    if diagnostics else pd.DataFrame()
)

if not frames:
    raise RuntimeError(
        "No usable rows produced. Inspect the messages above or increase MAX_OVERPASSES."
    )

df = pd.concat(frames, ignore_index=True)

# Match the batch runner's Parquet column types.
string_columns = {
    "folder", "date", "beam_str", "resource", "site_id",
    "ATL03", "ATL08", "atl08_source", "atl08_segment_id",
}
for column in df:
    df[column] = (
        df[column].astype("string")
        if column in string_columns
        else pd.to_numeric(df[column], errors="raise").astype("float64")
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
parquet = OUTPUT_DIR / f"{site.site_id}_small_test.parquet"
df.to_parquet(parquet, index=False, engine="pyarrow", compression="zstd")

df = pd.read_parquet(parquet)
print(f"\nReloaded {len(df):,} rows from {parquet}")
display(df.head(20))
display(df.groupby("resource").size().rename("rows").to_frame())
df.info()

Warning, this environment is using an outdated client (v5.6.3). The code will run but some functionality supported by the server (v5.8.0) may not be available.


Discovering overpasses for S01: 2020-01-01T00:00:00Z to 2020-08-01T00:00:00Z
Found 163 acquisitions; testing 3.

Overpass 1/3: ATL03_20200101021201_00820603_007_01.h5
  gt1l: ATL08=1315 msw=2.065 layer=0.720


2026-11-02 16:45:47,685 - INFO - Starting proxy for atl03x to process 1 resource(s) with 1 thread(s)
2026-11-02 16:45:48,668 - INFO - (dataframe.lua:138) <AppServer.50586> ATL03_20200101021201_00820603_007_01.h5/gt1l generated 744984 rows and 18 columns
2026-11-02 16:45:48,670 - INFO - Successfully completed processing resource [1 out of 1]: ATL03_20200101021201_00820603_007_01.h5
2026-11-02 16:45:48,670 - INFO - Successfuly received 744984 x 24 dataframe (eofs=1, dups=0, ptls=0)
2026-11-02 16:45:49,679 - INFO - Writing output file: /tmp/tmpjd5qdw7n
2026-11-02 16:45:52,810 - INFO - Closing output file: /tmp/tmpjd5qdw7n


  gt1r: ATL08=2135 msw=2.343 layer=0.801


2026-11-02 16:46:16,489 - INFO - Starting proxy for atl03x to process 1 resource(s) with 1 thread(s)
2026-11-02 16:46:23,159 - INFO - (dataframe.lua:138) <AppServer.50635> ATL03_20200101021201_00820603_007_01.h5/gt1r generated 2998716 rows and 18 columns
2026-11-02 16:46:23,159 - INFO - Successfully completed processing resource [1 out of 1]: ATL03_20200101021201_00820603_007_01.h5
2026-11-02 16:46:23,161 - INFO - Successfuly received 2998716 x 24 dataframe (eofs=1, dups=0, ptls=0)
2026-11-02 16:46:27,368 - INFO - Writing output file: /tmp/tmpi0takg98
2026-11-02 16:46:33,912 - INFO - Closing output file: /tmp/tmpi0takg98


  gt2l: ATL08=1221 msw=1.882 layer=0.660


... (36 outputs left out)

Retained rows: 0

Reloaded 21,255 rows from /home/s1803229/scratch/s1803229/ATL08_parquet_test/S01_small_test.parquet


,folder,date,lon,lat,pvpg,pv,pg,Eg,Ev,data_quantity,...,layer_flag,bin_id,atl08_segment_id,atl08_parent_count,atl08_parent_fraction,atl08_matched_photon_fraction,resource,site_id,length_m,canopy_openness_resolution_m
0,S01,01/01/2020,-120.906099,61.167568,0.540578,1.929137,3.568657,1.538462,1.153846,8.0,...,0.0,227238.0,721490081558609,1.0,1.000000,1.00000,ATL03_20200101021201_00820603_007_01.h5,S01,30.0,100.0
1,S01,01/01/2020,-120.906099,61.167568,0.540578,1.929137,3.568657,1.631579,0.973684,8.0,...,0.0,227239.0,721490081558609,1.0,1.000000,1.00000,ATL03_20200101021201_00820603_007_01.h5,S01,30.0,100.0
2,S01,01/01/2020,-120.906099,61.167568,0.540578,1.929137,3.568657,1.378378,1.216216,8.0,...,0.0,227240.0,721490081558609,1.0,1.000000,1.00000,ATL03_20200101021201_00820603_007_01.h5,S01,30.0,100.0
3,S01,01/01/2020,-120.906099,61.167568,0.540578,1.929137,3.568657,1.594595,1.081081,8.0,...,0.0,227241.0,721490081558614,2.0,0.947761,1.00000,ATL03_20200101021201_00820603_007_01.h5,S01,30.0,100.0
4,S01,01/01/2020,-120.906099,61.167568,0.540578,1.929137,3.568657,2.324324,0.729730,8.0,...,0.0,227242.0,721490081558614,1.0,1.000000,1.00000,ATL03_20200101021201_00820603_007_01.h5,S01,30.0,100.0
5,S01,01/01/2020,-120.906099,61.167568,0.540578,1.929137,3.568657,1.756098,0.902439,8.0,...,0.0,227243.0,721490081558614,1.0,1.000000,1.00000,ATL03_20200101021201_00820603_007_01.h5,S01,30.0,100.0
6,S01,01/01/2020,-120.906099,61.167568,0.540578,1.929137,3.568657,1.891892,0.810811,8.0,...,0.0,227244.0,721490081558619,2.0,0.689655,1.00000,ATL03_20200101021201_00820603_007_01.h5,S01,30.0,100.0
7,S01,01/01/2020,-120.906099,61.167568,0.540578,1.929137,3.568657,1.974359,0.948718,8.0,...,0.0,227245.0,721490081558619,1.0,1.000000,1.00000,ATL03_20200101021201_00820603_007_01.h5,S01,30.0,100.0
8,S01,01/01/2020,-120.906099,61.167568,0.540578,0.969338,1.793151,0.608696,0.782609,5.0,...,0.0,227238.0,721485786591313,1.0,1.000000,1.00000,ATL03_20200101021201_00820603_007_01.h5,S01,30.0,100.0
9,S01,01/01/2020,-120.906099,61.167568,0.540578,0.969338,1.793151,0.714286,0.476190,5.0,...,0.0,227239.0,721485786591313,1.0,1.000000,1.00000,ATL03_20200101021201_00820603_007_01.h5,S01,30.0,100.0


,rows
resource,
ATL03_20200101021201_00820603_007_01.h5,6929
ATL03_20200101145654_00900605_007_01.h5,14326


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21255 entries, 0 to 21254
Data columns (total 39 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   folder                         21255 non-null  string 
 1   date                           21255 non-null  string 
 2   lon                            21255 non-null  float64
 3   lat                            21255 non-null  float64
 4   pvpg                           21255 non-null  float64
 5   pv                             21255 non-null  float64
 6   pg                             21255 non-null  float64
 7   Eg                             21255 non-null  float64
 8   Ev                             21255 non-null  float64
 9   data_quantity                  21255 non-null  float64
 10  altitude                       0 non-null      float64
 11  pv_ratio_mean                  20281 non-null  float64
 12  pv_ratio_max                   20281 non-null 

In [3]:
import pandas as pd
from IPython.display import display

df = pd.read_parquet("../../../scratch/s1803229/ATL08_parquet_test/S01_small_test.parquet")

print(f"{len(df):,} rows × {len(df.columns)} columns")
display(df.head(20))

21,255 rows × 39 columns


,folder,date,lon,lat,pvpg,pv,pg,Eg,Ev,data_quantity,...,layer_flag,bin_id,atl08_segment_id,atl08_parent_count,atl08_parent_fraction,atl08_matched_photon_fraction,resource,site_id,length_m,canopy_openness_resolution_m
0,S01,01/01/2020,-120.906099,61.167568,0.540578,1.929137,3.568657,1.538462,1.153846,8.0,...,0.0,227238.0,721490081558609,1.0,1.000000,1.00000,ATL03_20200101021201_00820603_007_01.h5,S01,30.0,100.0
1,S01,01/01/2020,-120.906099,61.167568,0.540578,1.929137,3.568657,1.631579,0.973684,8.0,...,0.0,227239.0,721490081558609,1.0,1.000000,1.00000,ATL03_20200101021201_00820603_007_01.h5,S01,30.0,100.0
2,S01,01/01/2020,-120.906099,61.167568,0.540578,1.929137,3.568657,1.378378,1.216216,8.0,...,0.0,227240.0,721490081558609,1.0,1.000000,1.00000,ATL03_20200101021201_00820603_007_01.h5,S01,30.0,100.0
3,S01,01/01/2020,-120.906099,61.167568,0.540578,1.929137,3.568657,1.594595,1.081081,8.0,...,0.0,227241.0,721490081558614,2.0,0.947761,1.00000,ATL03_20200101021201_00820603_007_01.h5,S01,30.0,100.0
4,S01,01/01/2020,-120.906099,61.167568,0.540578,1.929137,3.568657,2.324324,0.729730,8.0,...,0.0,227242.0,721490081558614,1.0,1.000000,1.00000,ATL03_20200101021201_00820603_007_01.h5,S01,30.0,100.0
5,S01,01/01/2020,-120.906099,61.167568,0.540578,1.929137,3.568657,1.756098,0.902439,8.0,...,0.0,227243.0,721490081558614,1.0,1.000000,1.00000,ATL03_20200101021201_00820603_007_01.h5,S01,30.0,100.0
6,S01,01/01/2020,-120.906099,61.167568,0.540578,1.929137,3.568657,1.891892,0.810811,8.0,...,0.0,227244.0,721490081558619,2.0,0.689655,1.00000,ATL03_20200101021201_00820603_007_01.h5,S01,30.0,100.0
7,S01,01/01/2020,-120.906099,61.167568,0.540578,1.929137,3.568657,1.974359,0.948718,8.0,...,0.0,227245.0,721490081558619,1.0,1.000000,1.00000,ATL03_20200101021201_00820603_007_01.h5,S01,30.0,100.0
8,S01,01/01/2020,-120.906099,61.167568,0.540578,0.969338,1.793151,0.608696,0.782609,5.0,...,0.0,227238.0,721485786591313,1.0,1.000000,1.00000,ATL03_20200101021201_00820603_007_01.h5,S01,30.0,100.0
9,S01,01/01/2020,-120.906099,61.167568,0.540578,0.969338,1.793151,0.714286,0.476190,5.0,...,0.0,227239.0,721485786591313,1.0,1.000000,1.00000,ATL03_20200101021201_00820603_007_01.h5,S01,30.0,100.0


In [5]:
df.iloc[0]

folder                                                               S01
date                                                          01/01/2020
lon                                                          -120.906099
lat                                                            61.167568
pvpg                                                            0.540578
pv                                                              1.929137
pg                                                              3.568657
Eg                                                              1.538462
Ev                                                              1.153846
data_quantity                                                        8.0
altitude                                                             NaN
pv_ratio_mean                                                    1.99016
pv_ratio_max                                                     1.99016
beam                                               